# 02 - Dataset2 Honeybee Patch Hazırlığı

Bu notebook, Dataset2 için YOLO etiketlerinden pozitif ve negatif patch metadata tablolarını üretir.

Ham görüntüler değiştirilmez. Üretilen patch metadata dosyaları sonraki özellik çıkarımı aşamasında kullanılır.


## 1. Kütüphaneler

Bu bölümde dosya işlemleri, tablo işlemleri ve görsel işlemler için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from PIL import Image, ImageDraw
except ImportError as exc:
    raise ImportError("Pillow gerekli. Kurulum: pip install pillow") from exc

warnings.filterwarnings("ignore")


## 2. Ayarlar

Dataset adı, patch boyutları, negatif örnekleme türleri ve overwrite davranışı burada tanımlanır.


In [ ]:
DATASET_NAME = "dataset2_honeybee"
DATASET_DIR_NAME = "dataset2_honeybee"
DATASET_SHORT_NAME = "dataset2"

STAGE_NAME = "02_patch_preparation"
NOTEBOOK_NAME = "02_dataset2_honeybee_patch_preparation"

SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

CLASS_MAP = {0: "varroa_mite"}
POSITIVE_CLASS_NAME_KEYWORD = "varroa"

PATCH_SIZES = [48, 64, 80]
RATIO_VARIANTS = ["balanced", "neg4x"]
NEGATIVE_TYPES = ["center_negative", "mixed_negative", "outer_negative"]

RANDOM_STATE = 42
MAX_NEGATIVE_SAMPLING_ATTEMPTS = 20
SANITY_EXAMPLES_PER_GRID = 12
MAX_BBOX_INSIDE_PATCH_RATIO = 0.01

OVERWRITE_PATCH_METADATA = False
OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

DISPLAY_PREVIEW_ROWS = 12

print("Ayarlar hazır.")
print("Dataset:", DATASET_NAME)
print("Patch boyutları:", PATCH_SIZES)
print("Oran varyantları:", RATIO_VARIANTS)


## 3. Yardımcı Fonksiyonlar ve Dosya Yolları

Proje kökü bulunur; ham veri, metadata, tablo ve görsel çıktı klasörleri hazırlanır.


In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        try:
            start_path = Path(__file__).resolve()
        except NameError:
            start_path = Path.cwd().resolve()

    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data" / "raw").exists() and (candidate / "approaches").exists():
            return candidate

    raise FileNotFoundError(
        "Proje kökü bulunamadı. Notebook'u proje klasörü içinde çalıştırdığından emin ol."
    )


PROJECT_ROOT = find_project_root()
APPROACH_NAME = "traditional_ml"
APPROACH_DIR = PROJECT_ROOT / "approaches" / APPROACH_NAME
NOTEBOOK_DIR = APPROACH_DIR / "notebooks" / STAGE_NAME
OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / DATASET_DIR_NAME
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / DATASET_NAME

for directory in [METADATA_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def relative_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def log_section(title):
    print()
    print(f"===== {title} =====")


def log_output(message, level="INFO"):
    print(f"[{level}] {message}")


def log_dataframe(title, df, max_rows=12, note=None):
    print()
    print(f"--- {title} ---")
    if note:
        print(note)
    if df is None:
        print("DataFrame yok.")
        return
    print(f"Shape: {df.shape}")
    display(df.head(max_rows))


def save_dataframe_csv(df, output_path, overwrite=True, file_type="table_csv", note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        try:
            loaded_df = pd.read_csv(output_path)
            log_output(f"[LOAD] Existing CSV loaded: {relative_path(output_path)}")
            return loaded_df, "loaded_existing"
        except pd.errors.EmptyDataError:
            log_output(
                f"[SKIP] Existing CSV is empty; current DataFrame kept in memory: {relative_path(output_path)}",
                level="WARNING",
            )
            return df, "kept_current_empty_existing"

    if output_path.exists() and overwrite:
        log_output(f"[OVERWRITE] Replacing CSV: {relative_path(output_path)}")

    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    log_output(f"[SAVE] CSV saved: {relative_path(output_path)}")
    return df, "created_or_overwritten"


def save_matplotlib_figure(fig, output_path, title, description=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not OVERWRITE_FIGURES:
        plt.close(fig)
        log_output(f"[SKIP] Existing figure kept: {relative_path(output_path)}")
        return output_path

    if output_path.exists() and OVERWRITE_FIGURES:
        log_output(f"[OVERWRITE] Replacing figure: {relative_path(output_path)}")

    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    log_output(f"[SAVE] Figure saved: {relative_path(output_path)}")
    return output_path


print("Proje kökü:", PROJECT_ROOT)
print("Ham veri:", RAW_DATA_DIR)
print("Metadata çıktısı:", METADATA_DIR)
print("Notebook çıktısı:", OUTPUT_DIR)

## 4. Veri Seti Envanteri

Split bazında görüntü ve etiket dosyalarının temel sayımları çıkarılır.


In [ ]:
log_section("01 PATCH PREPARATION BAŞLADI")

log_output(f"Çalıştırma zamanı: {RUN_TIMESTAMP}")
log_output(f"Proje kökü: {PROJECT_ROOT}")
log_output(f"Raw dataset: {RAW_DATA_DIR}")
log_output(f"Metadata output: {METADATA_DIR}")
log_output(f"Notebook output: {OUTPUT_DIR}")
log_output(f"Patch sizes: {PATCH_SIZES}")
log_output(f"Ratio variants: {RATIO_VARIANTS}")
log_output(f"Class map: {CLASS_MAP}")

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw dataset klasörü bulunamadı: {RAW_DATA_DIR}")

inventory_records = []

for split in SPLITS:
    image_dir = RAW_DATA_DIR / split / "images"
    label_dir = RAW_DATA_DIR / split / "labels"

    if not image_dir.exists():
        raise FileNotFoundError(f"Image klasörü bulunamadı: {image_dir}")
    if not label_dir.exists():
        raise FileNotFoundError(f"Label klasörü bulunamadı: {label_dir}")

    image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])
    label_paths = sorted([p for p in label_dir.iterdir() if p.suffix.lower() == ".txt"])

    image_stems = {p.stem for p in image_paths}
    label_stems = {p.stem for p in label_paths}

    for image_path in image_paths:
        label_path = label_dir / f"{image_path.stem}.txt"
        inventory_records.append({
            "split": split,
            "image_path": str(image_path),
            "image_file": image_path.name,
            "image_stem": image_path.stem,
            "label_path": str(label_path),
            "label_file": label_path.name,
            "image_exists": True,
            "label_exists": label_path.exists(),
            "label_size_bytes": label_path.stat().st_size if label_path.exists() else np.nan,
        })

    missing_image_stems = sorted(label_stems - image_stems)
    for stem in missing_image_stems:
        label_path = label_dir / f"{stem}.txt"
        inventory_records.append({
            "split": split,
            "image_path": "",
            "image_file": "",
            "image_stem": stem,
            "label_path": str(label_path),
            "label_file": label_path.name,
            "image_exists": False,
            "label_exists": True,
            "label_size_bytes": label_path.stat().st_size if label_path.exists() else np.nan,
        })

inventory_df = pd.DataFrame(inventory_records)

source_file_audit_by_split_df = (
    inventory_df
    .groupby("split", dropna=False)
    .agg(
        image_count=("image_exists", "sum"),
        label_count=("label_exists", "sum"),
        missing_label_count=("label_exists", lambda s: int((~s).sum())),
        missing_image_count=("image_exists", lambda s: int((~s).sum())),
        empty_label_file_count=("label_size_bytes", lambda s: int((s == 0).sum())),
    )
    .reset_index()
)

source_file_audit_by_split_df["unreadable_image_count"] = 0

source_file_audit_by_split_df, _ = save_dataframe_csv(
    source_file_audit_by_split_df,
    TABLES_DIR / "source_file_audit_by_split.csv",
    overwrite=OVERWRITE_TABLES,
    note="Split bazlı image/label kaynak kontrolü.",
)

log_dataframe("Source File Audit by Split", source_file_audit_by_split_df)


## 5. Görüntü ve Boş Etiket Kontrolleri

Görüntü boyutları, okunamayan dosyalar ve boş etiket dosyaları kontrol edilir.


In [ ]:
log_section("02 IMAGE SIZE VE EMPTY LABEL KONTROLLERİ")

image_size_records = []
unreadable_records = []

for _, row in inventory_df[inventory_df["image_exists"]].iterrows():
    image_path = Path(row["image_path"])

    try:
        with Image.open(image_path) as img:
            width, height = img.size

        image_size_records.append({
            "split": row["split"],
            "image_path": row["image_path"],
            "image_file": row["image_file"],
            "image_stem": row["image_stem"],
            "label_path": row["label_path"],
            "label_file": row["label_file"],
            "label_exists": row["label_exists"],
            "empty_label_file": bool(row["label_size_bytes"] == 0),
            "image_width": int(width),
            "image_height": int(height),
            "unreadable_image": False,
        })

    except Exception as exc:
        unreadable_records.append({
            "split": row["split"],
            "image_path": row["image_path"],
            "image_file": row["image_file"],
            "error": str(exc),
        })

        image_size_records.append({
            "split": row["split"],
            "image_path": row["image_path"],
            "image_file": row["image_file"],
            "image_stem": row["image_stem"],
            "label_path": row["label_path"],
            "label_file": row["label_file"],
            "label_exists": row["label_exists"],
            "empty_label_file": bool(row["label_size_bytes"] == 0),
            "image_width": np.nan,
            "image_height": np.nan,
            "unreadable_image": True,
        })

images_df = pd.DataFrame(image_size_records)
unreadable_images_df = pd.DataFrame(unreadable_records)

source_file_audit_by_split_df["unreadable_image_count"] = (
    images_df.groupby("split")["unreadable_image"].sum()
    .reindex(source_file_audit_by_split_df["split"])
    .fillna(0)
    .astype(int)
    .values
)

source_file_audit_by_split_df, _ = save_dataframe_csv(
    source_file_audit_by_split_df,
    TABLES_DIR / "source_file_audit_by_split.csv",
    overwrite=OVERWRITE_TABLES,
    note="Image size okuması sonrası güncellenmiş kaynak audit tablosu.",
)

if not unreadable_images_df.empty:
    unreadable_images_df, _ = save_dataframe_csv(
        unreadable_images_df,
        TABLES_DIR / "unreadable_images.csv",
        overwrite=OVERWRITE_TABLES,
        note="Okunamayan image kayıtları.",
    )

log_dataframe("Image Size Preview", images_df.head(12))
log_dataframe("Updated Source File Audit by Split", source_file_audit_by_split_df)


## 6. YOLO Etiket Ayrıştırma

Etiket satırları okunur, bbox koordinatları piksel düzeyine çevrilir ve geçersiz satırlar ayrılır.


In [ ]:
log_section("03 YOLO LABEL PARSE")

def parse_yolo_line(raw_line, row_index):
    fields = raw_line.strip().split()

    parsed = {
        "row_index": row_index,
        "raw_line": raw_line.strip(),
        "fields_count": len(fields),
        "class_id": np.nan,
        "class_name": "",
        "x_center_norm": np.nan,
        "y_center_norm": np.nan,
        "bbox_width_norm": np.nan,
        "bbox_height_norm": np.nan,
        "is_valid_yolo_row": False,
        "invalid_reason": "",
    }

    if len(fields) != 5:
        parsed["invalid_reason"] = "field_count_not_5"
        return parsed

    try:
        class_id_float = float(fields[0])
        x_center = float(fields[1])
        y_center = float(fields[2])
        bbox_width = float(fields[3])
        bbox_height = float(fields[4])
    except ValueError:
        parsed["invalid_reason"] = "non_numeric_value"
        return parsed

    if not class_id_float.is_integer():
        parsed["invalid_reason"] = "class_id_not_integer"
        return parsed

    class_id = int(class_id_float)

    parsed.update({
        "class_id": class_id,
        "class_name": CLASS_MAP.get(class_id, "unknown"),
        "x_center_norm": x_center,
        "y_center_norm": y_center,
        "bbox_width_norm": bbox_width,
        "bbox_height_norm": bbox_height,
    })

    if class_id not in CLASS_MAP:
        parsed["invalid_reason"] = "unknown_class_id"
        return parsed

    if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
        parsed["invalid_reason"] = "center_out_of_range"
        return parsed

    if not (0 < bbox_width <= 1 and 0 < bbox_height <= 1):
        parsed["invalid_reason"] = "bbox_size_out_of_range"
        return parsed

    parsed["is_valid_yolo_row"] = True
    parsed["invalid_reason"] = ""
    return parsed


yolo_records = []

for _, image_row in images_df.iterrows():
    label_path = Path(image_row["label_path"])

    if not bool(image_row["label_exists"]) or bool(image_row["empty_label_file"]):
        continue

    try:
        lines = label_path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        lines = label_path.read_text(encoding="latin-1").splitlines()

    for row_index, raw_line in enumerate(lines):
        if raw_line.strip() == "":
            continue

        parsed = parse_yolo_line(raw_line, row_index)

        base = {
            "dataset_name": DATASET_NAME,
            "split": image_row["split"],
            "source_image_path": image_row["image_path"],
            "source_label_path": image_row["label_path"],
            "image_file": image_row["image_file"],
            "label_file": image_row["label_file"],
            "image_width": image_row["image_width"],
            "image_height": image_row["image_height"],
            "empty_label_file": image_row["empty_label_file"],
            "unreadable_image": image_row["unreadable_image"],
        }

        yolo_records.append({**base, **parsed})

yolo_rows_df = pd.DataFrame(yolo_records)

if yolo_rows_df.empty:
    yolo_rows_df = pd.DataFrame(columns=[
        "dataset_name", "split", "source_image_path", "source_label_path", "image_file", "label_file",
        "image_width", "image_height", "empty_label_file", "unreadable_image", "row_index", "raw_line",
        "fields_count", "class_id", "class_name", "x_center_norm", "y_center_norm", "bbox_width_norm",
        "bbox_height_norm", "is_valid_yolo_row", "invalid_reason"
    ])

valid_yolo_rows_df = yolo_rows_df[yolo_rows_df["is_valid_yolo_row"] == True].copy()
invalid_yolo_rows_df = yolo_rows_df[yolo_rows_df["is_valid_yolo_row"] == False].copy()

for df in [yolo_rows_df, valid_yolo_rows_df, invalid_yolo_rows_df]:
    if not df.empty:
        df["bbox_x_center_px"] = df["x_center_norm"] * df["image_width"]
        df["bbox_y_center_px"] = df["y_center_norm"] * df["image_height"]
        df["bbox_width_px"] = df["bbox_width_norm"] * df["image_width"]
        df["bbox_height_px"] = df["bbox_height_norm"] * df["image_height"]
        df["bbox_area_norm"] = df["bbox_width_norm"] * df["bbox_height_norm"]
        df["bbox_area_px"] = df["bbox_width_px"] * df["bbox_height_px"]

yolo_row_audit_df = (
    yolo_rows_df
    .groupby("split", dropna=False)
    .agg(
        total_yolo_rows=("raw_line", "count"),
        valid_yolo_rows=("is_valid_yolo_row", "sum"),
        invalid_yolo_rows=("is_valid_yolo_row", lambda s: int((~s).sum())),
    )
    .reset_index()
)

yolo_row_audit_df, _ = save_dataframe_csv(
    yolo_row_audit_df,
    TABLES_DIR / "yolo_row_audit.csv",
    overwrite=OVERWRITE_TABLES,
    note="Split bazlı YOLO row parse özeti.",
)

if not invalid_yolo_rows_df.empty:
    invalid_yolo_rows_df, _ = save_dataframe_csv(
        invalid_yolo_rows_df,
        TABLES_DIR / "invalid_yolo_rows.csv",
        overwrite=OVERWRITE_TABLES,
        note="Invalid YOLO row detayları.",
    )

log_dataframe("YOLO Row Audit", yolo_row_audit_df)
if not invalid_yolo_rows_df.empty:
    log_dataframe("Invalid YOLO Rows", invalid_yolo_rows_df.head(20))


## 7. Nesne Sayımı ve Aykırı Değer Kontrolü

Görüntü başına Varroa sayıları ve bbox ölçüleri özetlenir.


In [ ]:
log_section("04 OBJECT COUNT VE OUTLIER ANALİZİ")

POSITIVE_CLASS_IDS = [
    class_id for class_id, class_name in CLASS_MAP.items()
    if POSITIVE_CLASS_NAME_KEYWORD.lower() in class_name.lower()
]

log_output(f"Positive class ids: {POSITIVE_CLASS_IDS}")

if valid_yolo_rows_df.empty:
    positive_candidate_rows_df = pd.DataFrame()
else:
    positive_candidate_rows_df = valid_yolo_rows_df[
        valid_yolo_rows_df["class_id"].isin(POSITIVE_CLASS_IDS)
    ].copy()

object_count_records = []

for _, image_row in images_df.iterrows():
    image_path = image_row["image_path"]
    image_valid_rows = valid_yolo_rows_df[
        valid_yolo_rows_df["source_image_path"] == image_path
    ] if not valid_yolo_rows_df.empty else pd.DataFrame()

    image_positive_rows = image_valid_rows[
        image_valid_rows["class_id"].isin(POSITIVE_CLASS_IDS)
    ] if not image_valid_rows.empty else pd.DataFrame()

    image_invalid_rows = invalid_yolo_rows_df[
        invalid_yolo_rows_df["source_image_path"] == image_path
    ] if not invalid_yolo_rows_df.empty else pd.DataFrame()

    object_count_records.append({
        "dataset_name": DATASET_NAME,
        "split": image_row["split"],
        "source_image_path": image_path,
        "source_label_path": image_row["label_path"],
        "image_file": image_row["image_file"],
        "image_width": image_row["image_width"],
        "image_height": image_row["image_height"],
        "empty_label_file": bool(image_row["empty_label_file"]),
        "unreadable_image": bool(image_row["unreadable_image"]),
        "valid_yolo_object_count": int(len(image_valid_rows)),
        "varroa_count": int(len(image_positive_rows)),
        "image_has_invalid_yolo_row": bool(len(image_invalid_rows) > 0),
    })

image_object_counts_df = pd.DataFrame(object_count_records)

METRICS_FOR_OUTLIER = [
    "bbox_width_norm",
    "bbox_height_norm",
    "bbox_area_norm",
    "bbox_width_px",
    "bbox_height_px",
    "bbox_area_px",
]


def compute_outlier_thresholds(df, metrics):
    threshold_records = []

    for metric in metrics:
        series = pd.to_numeric(df[metric], errors="coerce").dropna()

        if series.empty:
            small_threshold = np.nan
            large_threshold = np.nan
            q1 = q3 = iqr = q005 = q995 = np.nan
        else:
            q1 = float(series.quantile(0.25))
            q3 = float(series.quantile(0.75))
            iqr = q3 - q1
            q005 = float(series.quantile(0.005))
            q995 = float(series.quantile(0.995))
            small_threshold = max(0, min(q005, q1 - 3 * iqr))
            large_threshold = max(q995, q3 + 3 * iqr)

        threshold_records.append({
            "metric": metric,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "q005": q005,
            "q995": q995,
            "small_threshold": small_threshold,
            "large_threshold": large_threshold,
        })

    return pd.DataFrame(threshold_records)


outlier_thresholds_df = compute_outlier_thresholds(positive_candidate_rows_df, METRICS_FOR_OUTLIER)

positive_candidate_rows_df["is_extreme_small_candidate"] = False
positive_candidate_rows_df["is_extreme_large_candidate"] = False

for _, threshold_row in outlier_thresholds_df.iterrows():
    metric = threshold_row["metric"]
    small_threshold = threshold_row["small_threshold"]
    large_threshold = threshold_row["large_threshold"]

    if pd.notna(small_threshold):
        positive_candidate_rows_df[f"{metric}_small_outlier"] = positive_candidate_rows_df[metric] <= small_threshold
        positive_candidate_rows_df["is_extreme_small_candidate"] = (
            positive_candidate_rows_df["is_extreme_small_candidate"]
            | positive_candidate_rows_df[f"{metric}_small_outlier"]
        )

    if pd.notna(large_threshold):
        positive_candidate_rows_df[f"{metric}_large_outlier"] = positive_candidate_rows_df[metric] >= large_threshold
        positive_candidate_rows_df["is_extreme_large_candidate"] = (
            positive_candidate_rows_df["is_extreme_large_candidate"]
            | positive_candidate_rows_df[f"{metric}_large_outlier"]
        )

positive_candidate_rows_df["is_bbox_exclusion_candidate"] = (
    positive_candidate_rows_df["is_extreme_small_candidate"]
    | positive_candidate_rows_df["is_extreme_large_candidate"]
)

accepted_positive_rows_df = positive_candidate_rows_df[
    positive_candidate_rows_df["is_bbox_exclusion_candidate"] == False
].copy()

bbox_exclusion_candidates_df = positive_candidate_rows_df[
    positive_candidate_rows_df["is_bbox_exclusion_candidate"] == True
].copy()

outlier_thresholds_df, _ = save_dataframe_csv(
    outlier_thresholds_df,
    TABLES_DIR / "outlier_thresholds.csv",
    overwrite=OVERWRITE_TABLES,
    note="Dataset-specific symmetric outlier thresholds.",
)

log_dataframe("Outlier Thresholds", outlier_thresholds_df)
log_output(f"Positive candidate bbox count: {len(positive_candidate_rows_df)}")
log_output(f"Accepted positive bbox count: {len(accepted_positive_rows_df)}")
log_output(f"BBox exclusion candidate count: {len(bbox_exclusion_candidates_df)}")


## 8. Pozitif ve Negatif Kaynak Kontrolü

Patch üretiminde kullanılacak pozitif bbox kayıtları ve negatif kaynak görüntüler hazırlanır.


In [ ]:
log_section("05 POSITIVE / NEGATIVE SOURCE AUDIT")

if DATASET_NAME == "dataset1_varroa":
    negative_source_mask = (
        (image_object_counts_df["varroa_count"] == 0)
        & (image_object_counts_df["empty_label_file"] == False)
        & (image_object_counts_df["unreadable_image"] == False)
    )
else:
    negative_source_mask = (
        (image_object_counts_df["varroa_count"] == 0)
        & (image_object_counts_df["image_has_invalid_yolo_row"] == False)
        & (image_object_counts_df["unreadable_image"] == False)
    )

negative_source_images_df = image_object_counts_df[negative_source_mask].copy()

positive_source_audit_records = []
negative_source_audit_records = []

for split in SPLITS:
    split_positive_candidates = positive_candidate_rows_df[positive_candidate_rows_df["split"] == split]
    split_accepted_positives = accepted_positive_rows_df[accepted_positive_rows_df["split"] == split]
    split_bbox_exclusions = bbox_exclusion_candidates_df[bbox_exclusion_candidates_df["split"] == split]

    positive_source_audit_records.append({
        "dataset_name": DATASET_NAME,
        "split": split,
        "positive_candidate_bbox_count": int(len(split_positive_candidates)),
        "accepted_positive_bbox_count": int(len(split_accepted_positives)),
        "excluded_bbox_outlier_count": int(len(split_bbox_exclusions)),
        "invalid_yolo_row_count": int(len(invalid_yolo_rows_df[invalid_yolo_rows_df["split"] == split])) if not invalid_yolo_rows_df.empty else 0,
    })

    split_images = image_object_counts_df[image_object_counts_df["split"] == split]
    split_negative_source_images = negative_source_images_df[negative_source_images_df["split"] == split]

    negative_source_audit_records.append({
        "dataset_name": DATASET_NAME,
        "split": split,
        "zero_varroa_image_count": int((split_images["varroa_count"] == 0).sum()),
        "empty_label_file_count": int(split_images["empty_label_file"].sum()),
        "invalid_yolo_row_image_count": int(split_images["image_has_invalid_yolo_row"].sum()),
        "accepted_negative_source_image_count": int(len(split_negative_source_images)),
    })

positive_source_audit_df = pd.DataFrame(positive_source_audit_records)
negative_source_audit_df = pd.DataFrame(negative_source_audit_records)

excluded_records = []

for _, row in bbox_exclusion_candidates_df.iterrows():
    excluded_records.append({
        "dataset_name": DATASET_NAME,
        "split": row["split"],
        "source_image_path": row["source_image_path"],
        "source_label_path": row["source_label_path"],
        "record_type": "bbox",
        "exclusion_level": "bbox",
        "exclusion_reason": "dataset_specific_extreme_bbox_candidate",
        "row_index": row["row_index"],
        "class_id": row["class_id"],
        "class_name": row["class_name"],
        "bbox_x_center_norm": row["x_center_norm"],
        "bbox_y_center_norm": row["y_center_norm"],
        "bbox_width_norm": row["bbox_width_norm"],
        "bbox_height_norm": row["bbox_height_norm"],
        "bbox_area_px": row["bbox_area_px"],
        "notes": "Excluded from positive patch generation.",
    })

for _, row in invalid_yolo_rows_df.iterrows():
    excluded_records.append({
        "dataset_name": DATASET_NAME,
        "split": row["split"],
        "source_image_path": row["source_image_path"],
        "source_label_path": row["source_label_path"],
        "record_type": "yolo_row",
        "exclusion_level": "row",
        "exclusion_reason": row["invalid_reason"],
        "row_index": row["row_index"],
        "class_id": row["class_id"],
        "class_name": row["class_name"],
        "bbox_x_center_norm": row["x_center_norm"],
        "bbox_y_center_norm": row["y_center_norm"],
        "bbox_width_norm": row["bbox_width_norm"],
        "bbox_height_norm": row["bbox_height_norm"],
        "bbox_area_px": row["bbox_area_px"] if "bbox_area_px" in row else np.nan,
        "notes": "Invalid YOLO row excluded from patch generation.",
    })

if DATASET_NAME == "dataset1_varroa":
    dataset1_empty_label_images = image_object_counts_df[image_object_counts_df["empty_label_file"] == True].copy()
    for _, row in dataset1_empty_label_images.iterrows():
        excluded_records.append({
            "dataset_name": DATASET_NAME,
            "split": row["split"],
            "source_image_path": row["source_image_path"],
            "source_label_path": row["source_label_path"],
            "record_type": "image",
            "exclusion_level": "image",
            "exclusion_reason": "empty_label_file_dataset1",
            "row_index": np.nan,
            "class_id": np.nan,
            "class_name": "",
            "bbox_x_center_norm": np.nan,
            "bbox_y_center_norm": np.nan,
            "bbox_width_norm": np.nan,
            "bbox_height_norm": np.nan,
            "bbox_area_px": np.nan,
            "notes": "Dataset1 empty label image excluded from negative source pool.",
        })

excluded_records_audit_df = pd.DataFrame(excluded_records)

positive_source_audit_df, _ = save_dataframe_csv(
    positive_source_audit_df,
    TABLES_DIR / "positive_source_audit.csv",
    overwrite=OVERWRITE_TABLES,
    note="Positive source bbox audit.",
)

negative_source_audit_df, _ = save_dataframe_csv(
    negative_source_audit_df,
    TABLES_DIR / "negative_source_audit.csv",
    overwrite=OVERWRITE_TABLES,
    note="Negative source image audit.",
)

excluded_records_audit_df, _ = save_dataframe_csv(
    excluded_records_audit_df,
    TABLES_DIR / "excluded_records_audit.csv",
    overwrite=OVERWRITE_TABLES,
    note="Excluded row/bbox/image records.",
)

log_dataframe("Positive Source Audit", positive_source_audit_df)
log_dataframe("Negative Source Audit", negative_source_audit_df)
log_dataframe("Excluded Records Audit", excluded_records_audit_df, max_rows=20)


## 9. Patch Yardımcı Fonksiyonları

Pozitif patch, negatif patch ve crop sınırı hesaplamaları için yardımcı fonksiyonlar tanımlanır.


In [ ]:
log_section("06 PATCH HELPER FONKSİYONLARI")

PATCH_METADATA_COLUMNS = [
    "patch_id",
    "dataset_name",
    "split",
    "source_image_path",
    "source_label_path",
    "patch_set",
    "patch_size",
    "ratio_variant",
    "patch_label",
    "patch_type",
    "x1",
    "y1",
    "x2",
    "y2",
    "crop_width",
    "crop_height",
    "image_width",
    "image_height",
    "patch_center_x_px",
    "patch_center_y_px",
    "requested_center_x_px",
    "requested_center_y_px",
    "was_shifted_to_fit_image",
    "shift_x_px",
    "shift_y_px",
    "source_bbox_id",
    "source_bbox_row_index",
    "source_bbox_class_id",
    "source_bbox_class_name",
    "source_bbox_x_center_norm",
    "source_bbox_y_center_norm",
    "source_bbox_width_norm",
    "source_bbox_height_norm",
    "source_bbox_x_center_px",
    "source_bbox_y_center_px",
    "source_bbox_width_px",
    "source_bbox_height_px",
    "source_bbox_area_px",
    "is_valid_yolo_row",
    "is_extreme_small_candidate",
    "is_extreme_large_candidate",
    "is_bbox_exclusion_candidate",
    "negative_type",
    "requested_negative_zone",
    "actual_negative_zone",
    "actual_normalized_center_distance",
    "negative_source_mode",
    "max_bbox_inside_patch_ratio",
    "overlaps_varroa_bbox",
    "sampling_seed",
    "sampling_attempt_index",
    "notes",
    "flags",
]

MAX_BBOX_INSIDE_PATCH_RATIO = 0.01


def make_patch_id(dataset_name, patch_set, split, patch_type, index_value):
    return f"{dataset_name}__{patch_set}__{split}__{patch_type}__{index_value:08d}"


def compute_crop_from_center(center_x, center_y, patch_size, image_width, image_height):
    if patch_size > image_width or patch_size > image_height:
        return None

    requested_x1 = int(round(center_x - patch_size / 2))
    requested_y1 = int(round(center_y - patch_size / 2))

    x1 = max(0, min(requested_x1, int(image_width - patch_size)))
    y1 = max(0, min(requested_y1, int(image_height - patch_size)))

    x2 = x1 + patch_size
    y2 = y1 + patch_size

    patch_center_x = x1 + patch_size / 2
    patch_center_y = y1 + patch_size / 2

    shift_x = x1 - requested_x1
    shift_y = y1 - requested_y1

    return {
        "x1": int(x1),
        "y1": int(y1),
        "x2": int(x2),
        "y2": int(y2),
        "crop_width": int(patch_size),
        "crop_height": int(patch_size),
        "patch_center_x_px": float(patch_center_x),
        "patch_center_y_px": float(patch_center_y),
        "requested_center_x_px": float(center_x),
        "requested_center_y_px": float(center_y),
        "was_shifted_to_fit_image": bool(shift_x != 0 or shift_y != 0),
        "shift_x_px": int(shift_x),
        "shift_y_px": int(shift_y),
    }


def normalized_center_distance(center_x, center_y, image_width, image_height):
    image_center_x = image_width / 2
    image_center_y = image_height / 2

    numerator = math.sqrt((center_x - image_center_x) ** 2 + (center_y - image_center_y) ** 2)
    denominator = math.sqrt(image_center_x ** 2 + image_center_y ** 2)

    if denominator == 0:
        return 0

    return float(numerator / denominator)


def zone_from_distance(distance):
    if distance < 0.25:
        return "center"
    if distance < 0.50:
        return "mixed"
    return "outer"


def negative_type_to_zone(negative_type):
    return negative_type.replace("_negative", "")


def zone_matches(distance, requested_zone):
    return zone_from_distance(distance) == requested_zone


def deterministic_center_for_zone(requested_zone, image_width, image_height):
    """Fallback center used when random rejection sampling does not hit the requested zone."""
    if requested_zone == "center":
        return image_width / 2, image_height / 2
    if requested_zone == "mixed":
        return image_width * 0.70, image_height / 2
    return image_width * 0.08, image_height * 0.08


def sample_center_for_requested_zone(rng, image_width, image_height, requested_zone):
    for _ in range(MAX_NEGATIVE_SAMPLING_ATTEMPTS):
        center_x = float(rng.uniform(0, image_width))
        center_y = float(rng.uniform(0, image_height))
        distance = normalized_center_distance(center_x, center_y, image_width, image_height)
        if zone_matches(distance, requested_zone):
            return center_x, center_y

    return deterministic_center_for_zone(requested_zone, image_width, image_height)


def bbox_to_xyxy(row):
    cx = float(row["bbox_x_center_px"])
    cy = float(row["bbox_y_center_px"])
    bw = float(row["bbox_width_px"])
    bh = float(row["bbox_height_px"])
    return {
        "x1": cx - bw / 2,
        "y1": cy - bh / 2,
        "x2": cx + bw / 2,
        "y2": cy + bh / 2,
        "area": max(0.0, bw * bh),
    }


varroa_bboxes_by_image = {}
if not positive_candidate_rows_df.empty:
    for image_path, group_df in positive_candidate_rows_df.groupby("source_image_path"):
        varroa_bboxes_by_image[image_path] = [bbox_to_xyxy(row) for _, row in group_df.iterrows()]


def max_bbox_inside_patch_ratio_for_image(image_path, crop):
    bboxes = varroa_bboxes_by_image.get(str(image_path), [])
    if not bboxes:
        return 0.0

    max_ratio = 0.0
    for bbox in bboxes:
        if bbox["area"] <= 0:
            continue

        inter_x1 = max(float(crop["x1"]), bbox["x1"])
        inter_y1 = max(float(crop["y1"]), bbox["y1"])
        inter_x2 = min(float(crop["x2"]), bbox["x2"])
        inter_y2 = min(float(crop["y2"]), bbox["y2"])

        inter_w = max(0.0, inter_x2 - inter_x1)
        inter_h = max(0.0, inter_y2 - inter_y1)
        inter_area = inter_w * inter_h
        inside_ratio = inter_area / bbox["area"]
        max_ratio = max(max_ratio, inside_ratio)

    return float(max_ratio)


def build_positive_patch_row(row, patch_set, patch_size, ratio_variant, patch_index):
    crop = compute_crop_from_center(
        center_x=row["bbox_x_center_px"],
        center_y=row["bbox_y_center_px"],
        patch_size=patch_size,
        image_width=row["image_width"],
        image_height=row["image_height"],
    )

    if crop is None:
        return None

    source_bbox_id = (
        f"{DATASET_NAME}__{row['split']}__"
        f"{Path(row['source_image_path']).stem}__row{int(row['row_index'])}"
    )

    patch_row = {
        "patch_id": make_patch_id(DATASET_NAME, patch_set, row["split"], "positive", patch_index),
        "dataset_name": DATASET_NAME,
        "split": row["split"],
        "source_image_path": row["source_image_path"],
        "source_label_path": row["source_label_path"],
        "patch_set": patch_set,
        "patch_size": patch_size,
        "ratio_variant": ratio_variant,
        "patch_label": 1,
        "patch_type": "positive",
        "image_width": int(row["image_width"]),
        "image_height": int(row["image_height"]),
        "source_bbox_id": source_bbox_id,
        "source_bbox_row_index": int(row["row_index"]),
        "source_bbox_class_id": int(row["class_id"]),
        "source_bbox_class_name": row["class_name"],
        "source_bbox_x_center_norm": row["x_center_norm"],
        "source_bbox_y_center_norm": row["y_center_norm"],
        "source_bbox_width_norm": row["bbox_width_norm"],
        "source_bbox_height_norm": row["bbox_height_norm"],
        "source_bbox_x_center_px": row["bbox_x_center_px"],
        "source_bbox_y_center_px": row["bbox_y_center_px"],
        "source_bbox_width_px": row["bbox_width_px"],
        "source_bbox_height_px": row["bbox_height_px"],
        "source_bbox_area_px": row["bbox_area_px"],
        "is_valid_yolo_row": True,
        "is_extreme_small_candidate": bool(row["is_extreme_small_candidate"]),
        "is_extreme_large_candidate": bool(row["is_extreme_large_candidate"]),
        "is_bbox_exclusion_candidate": bool(row["is_bbox_exclusion_candidate"]),
        "negative_type": "",
        "requested_negative_zone": "",
        "actual_negative_zone": "",
        "actual_normalized_center_distance": np.nan,
        "negative_source_mode": "",
        "max_bbox_inside_patch_ratio": np.nan,
        "overlaps_varroa_bbox": np.nan,
        "sampling_seed": RANDOM_STATE,
        "sampling_attempt_index": 0,
        "notes": "",
        "flags": "",
    }

    patch_row.update(crop)
    return patch_row


def split_count_across_negative_types(total_negative_count):
    base = total_negative_count // len(NEGATIVE_TYPES)
    remainder = total_negative_count % len(NEGATIVE_TYPES)

    counts = {}
    for index, negative_type in enumerate(NEGATIVE_TYPES):
        counts[negative_type] = base + (1 if index < remainder else 0)

    return counts


def build_negative_candidate(
    image_row,
    split,
    patch_set,
    patch_size,
    ratio_variant,
    negative_type,
    patch_index,
    rng,
    source_mode,
    require_non_overlap,
    attempt_index,
):
    image_width = int(image_row["image_width"])
    image_height = int(image_row["image_height"])

    if patch_size > image_width or patch_size > image_height:
        return None

    requested_zone = negative_type_to_zone(negative_type)
    center_x, center_y = sample_center_for_requested_zone(rng, image_width, image_height, requested_zone)

    crop = compute_crop_from_center(
        center_x,
        center_y,
        patch_size,
        image_width,
        image_height,
    )

    if crop is None:
        return None

    max_inside_ratio = max_bbox_inside_patch_ratio_for_image(image_row["source_image_path"], crop)
    overlaps_varroa_bbox = bool(max_inside_ratio >= MAX_BBOX_INSIDE_PATCH_RATIO)

    if require_non_overlap and overlaps_varroa_bbox:
        return None

    actual_distance = normalized_center_distance(
        crop["patch_center_x_px"],
        crop["patch_center_y_px"],
        image_width,
        image_height,
    )

    flags = []
    if crop["was_shifted_to_fit_image"]:
        flags.append("shifted_to_fit_image")
    if source_mode == "fallback_varroa_image_non_overlap":
        flags.append("varroa_image_non_overlap_fallback")

    candidate = {
        "patch_id": make_patch_id(DATASET_NAME, patch_set, split, negative_type, patch_index),
        "dataset_name": DATASET_NAME,
        "split": split,
        "source_image_path": image_row["source_image_path"],
        "source_label_path": image_row["source_label_path"],
        "patch_set": patch_set,
        "patch_size": patch_size,
        "ratio_variant": ratio_variant,
        "patch_label": 0,
        "patch_type": negative_type,
        "image_width": image_width,
        "image_height": image_height,
        "source_bbox_id": "",
        "source_bbox_row_index": np.nan,
        "source_bbox_class_id": np.nan,
        "source_bbox_class_name": "",
        "source_bbox_x_center_norm": np.nan,
        "source_bbox_y_center_norm": np.nan,
        "source_bbox_width_norm": np.nan,
        "source_bbox_height_norm": np.nan,
        "source_bbox_x_center_px": np.nan,
        "source_bbox_y_center_px": np.nan,
        "source_bbox_width_px": np.nan,
        "source_bbox_height_px": np.nan,
        "source_bbox_area_px": np.nan,
        "is_valid_yolo_row": np.nan,
        "is_extreme_small_candidate": np.nan,
        "is_extreme_large_candidate": np.nan,
        "is_bbox_exclusion_candidate": np.nan,
        "negative_type": negative_type,
        "requested_negative_zone": requested_zone,
        "actual_negative_zone": zone_from_distance(actual_distance),
        "actual_normalized_center_distance": actual_distance,
        "negative_source_mode": source_mode,
        "max_bbox_inside_patch_ratio": max_inside_ratio,
        "overlaps_varroa_bbox": overlaps_varroa_bbox,
        "sampling_seed": RANDOM_STATE,
        "sampling_attempt_index": attempt_index,
        "notes": "",
        "flags": ";".join(flags),
    }

    candidate.update(crop)
    return candidate


def sample_from_source_pool(
    source_images_df,
    split,
    patch_set,
    patch_size,
    ratio_variant,
    negative_type,
    patch_index,
    rng,
    used_duplicate_keys,
    source_mode,
    require_non_overlap,
    allow_duplicate,
):
    if source_images_df.empty:
        return None

    last_duplicate_candidate = None

    for attempt_index in range(1, MAX_NEGATIVE_SAMPLING_ATTEMPTS + 1):
        image_row = source_images_df.sample(
            n=1,
            random_state=int(rng.integers(0, 2**31 - 1)),
        ).iloc[0]

        candidate = build_negative_candidate(
            image_row=image_row,
            split=split,
            patch_set=patch_set,
            patch_size=patch_size,
            ratio_variant=ratio_variant,
            negative_type=negative_type,
            patch_index=patch_index,
            rng=rng,
            source_mode=source_mode,
            require_non_overlap=require_non_overlap,
            attempt_index=attempt_index,
        )

        if candidate is None:
            continue

        duplicate_key = (
            candidate["source_image_path"],
            candidate["patch_size"],
            candidate["split"],
            candidate["x1"],
            candidate["y1"],
            candidate["x2"],
            candidate["y2"],
        )

        if duplicate_key not in used_duplicate_keys:
            used_duplicate_keys.add(duplicate_key)
            return candidate

        last_duplicate_candidate = candidate

        if allow_duplicate:
            duplicate_flags = str(candidate.get("flags", "")).split(";") if candidate.get("flags", "") else []
            duplicate_flags.append("duplicate_crop_fallback")
            candidate["flags"] = ";".join([flag for flag in duplicate_flags if flag])
            return candidate

    if allow_duplicate and last_duplicate_candidate is not None:
        duplicate_flags = str(last_duplicate_candidate.get("flags", "")).split(";") if last_duplicate_candidate.get("flags", "") else []
        duplicate_flags.append("duplicate_crop_fallback")
        duplicate_flags.append("after_max_attempts")
        last_duplicate_candidate["flags"] = ";".join([flag for flag in duplicate_flags if flag])
        return last_duplicate_candidate

    return None


def sample_negative_patch(
    primary_source_images_df,
    fallback_source_images_df,
    split,
    patch_set,
    patch_size,
    ratio_variant,
    negative_type,
    patch_index,
    rng,
    used_duplicate_keys,
):
    # 1) Primary: no-varroa image source, unique crop preferred.
    candidate = sample_from_source_pool(
        source_images_df=primary_source_images_df,
        split=split,
        patch_set=patch_set,
        patch_size=patch_size,
        ratio_variant=ratio_variant,
        negative_type=negative_type,
        patch_index=patch_index,
        rng=rng,
        used_duplicate_keys=used_duplicate_keys,
        source_mode="primary_no_varroa_image",
        require_non_overlap=False,
        allow_duplicate=False,
    )
    if candidate is not None:
        return candidate

    # 2) Fallback: varroa image, but crop must not include Varroa bbox more than threshold.
    candidate = sample_from_source_pool(
        source_images_df=fallback_source_images_df,
        split=split,
        patch_set=patch_set,
        patch_size=patch_size,
        ratio_variant=ratio_variant,
        negative_type=negative_type,
        patch_index=patch_index,
        rng=rng,
        used_duplicate_keys=used_duplicate_keys,
        source_mode="fallback_varroa_image_non_overlap",
        require_non_overlap=True,
        allow_duplicate=False,
    )
    if candidate is not None:
        return candidate

    # 3) Last resort: allow duplicate crop from primary source and mark it in flags.
    candidate = sample_from_source_pool(
        source_images_df=primary_source_images_df,
        split=split,
        patch_set=patch_set,
        patch_size=patch_size,
        ratio_variant=ratio_variant,
        negative_type=negative_type,
        patch_index=patch_index,
        rng=rng,
        used_duplicate_keys=used_duplicate_keys,
        source_mode="primary_no_varroa_image_duplicate_fallback",
        require_non_overlap=False,
        allow_duplicate=True,
    )
    if candidate is not None:
        return candidate

    # 4) Extreme last resort: allow duplicate non-overlap fallback from varroa image.
    candidate = sample_from_source_pool(
        source_images_df=fallback_source_images_df,
        split=split,
        patch_set=patch_set,
        patch_size=patch_size,
        ratio_variant=ratio_variant,
        negative_type=negative_type,
        patch_index=patch_index,
        rng=rng,
        used_duplicate_keys=used_duplicate_keys,
        source_mode="fallback_varroa_image_non_overlap_duplicate_fallback",
        require_non_overlap=True,
        allow_duplicate=True,
    )
    return candidate


log_output("Patch helper fonksiyonları hazır.")
log_output(f"MAX_BBOX_INSIDE_PATCH_RATIO = {MAX_BBOX_INSIDE_PATCH_RATIO}")


## 10. Patch Metadata Üretimi

Belirlenen patch boyutları ve oran varyantları için metadata CSV dosyaları oluşturulur ya da mevcutsa okunur.


In [ ]:
log_section("07 PATCH METADATA ÜRETİMİ")

all_patch_metadata = {}
patch_metadata_manifest_records = []

for patch_size in PATCH_SIZES:
    for ratio_variant in RATIO_VARIANTS:
        patch_set = f"centered{patch_size}_{ratio_variant}"
        output_path = METADATA_DIR / f"patch_metadata_{patch_set}.csv"

        if output_path.exists() and not OVERWRITE_PATCH_METADATA:
            patch_df = pd.read_csv(output_path)
            all_patch_metadata[patch_set] = patch_df

            log_output(f"[LOAD] Existing patch metadata loaded: {output_path}")

        else:
            if output_path.exists() and OVERWRITE_PATCH_METADATA:
                log_output(f"[OVERWRITE] Existing patch metadata will be replaced: {output_path}")

            patch_rows = []
            patch_index = 0
            used_duplicate_keys = set()

            for split in SPLITS:
                split_positive_rows = accepted_positive_rows_df[
                    accepted_positive_rows_df["split"] == split
                ].copy()

                split_primary_negative_images = negative_source_images_df[
                    negative_source_images_df["split"] == split
                ].copy()

                split_fallback_negative_images = image_object_counts_df[
                    (image_object_counts_df["split"] == split)
                    & (image_object_counts_df["varroa_count"] > 0)
                    & (image_object_counts_df["unreadable_image"] == False)
                    & (image_object_counts_df["image_has_invalid_yolo_row"] == False)
                ].copy()

                split_positive_count = len(split_positive_rows)

                for _, positive_row in split_positive_rows.iterrows():
                    positive_patch = build_positive_patch_row(
                        positive_row,
                        patch_set=patch_set,
                        patch_size=patch_size,
                        ratio_variant=ratio_variant,
                        patch_index=patch_index,
                    )

                    if positive_patch is not None:
                        patch_rows.append(positive_patch)
                        patch_index += 1

                if ratio_variant == "balanced":
                    target_negative_count = split_positive_count
                elif ratio_variant == "neg4x":
                    target_negative_count = split_positive_count * 4
                else:
                    raise ValueError(f"Bilinmeyen ratio_variant: {ratio_variant}")

                negative_type_counts = split_count_across_negative_types(target_negative_count)

                log_output(
                    f"{patch_set} | {split}: positive={split_positive_count}, "
                    f"target_negative={target_negative_count}, "
                    f"primary_images={len(split_primary_negative_images)}, "
                    f"fallback_images={len(split_fallback_negative_images)}"
                )

                for negative_type, negative_count in negative_type_counts.items():
                    produced_for_type = 0

                    while produced_for_type < negative_count:
                        rng = np.random.default_rng(
                            RANDOM_STATE
                            + patch_size * 1000
                            + patch_index
                            + abs(hash((split, ratio_variant, negative_type, produced_for_type))) % 100000
                        )

                        negative_patch = sample_negative_patch(
                            primary_source_images_df=split_primary_negative_images,
                            fallback_source_images_df=split_fallback_negative_images,
                            split=split,
                            patch_set=patch_set,
                            patch_size=patch_size,
                            ratio_variant=ratio_variant,
                            negative_type=negative_type,
                            patch_index=patch_index,
                            rng=rng,
                            used_duplicate_keys=used_duplicate_keys,
                        )

                        if negative_patch is None:
                            raise RuntimeError(
                                f"Negative patch üretilemedi: patch_set={patch_set}, split={split}, "
                                f"negative_type={negative_type}, produced={produced_for_type}, target={negative_count}"
                            )

                        patch_rows.append(negative_patch)
                        patch_index += 1
                        produced_for_type += 1

            patch_df = pd.DataFrame(patch_rows)

            for column in PATCH_METADATA_COLUMNS:
                if column not in patch_df.columns:
                    patch_df[column] = np.nan

            patch_df = patch_df[PATCH_METADATA_COLUMNS].copy()
            patch_df = patch_df.sort_values(["split", "patch_label", "patch_type", "patch_id"]).reset_index(drop=True)

            patch_df, _ = save_dataframe_csv(
                patch_df,
                output_path,
                overwrite=OVERWRITE_PATCH_METADATA,
                file_type="patch_metadata_csv",
                note=f"Patch metadata for {patch_set}.",
            )

            all_patch_metadata[patch_set] = patch_df

        summary_agg = {
            "total_rows": ("patch_id", "count"),
            "positive_count": ("patch_label", lambda s: int((s == 1).sum())),
            "negative_count": ("patch_label", lambda s: int((s == 0).sum())),
            "center_negative_count": ("patch_type", lambda s: int((s == "center_negative").sum())),
            "mixed_negative_count": ("patch_type", lambda s: int((s == "mixed_negative").sum())),
            "outer_negative_count": ("patch_type", lambda s: int((s == "outer_negative").sum())),
            "shifted_crop_count": ("was_shifted_to_fit_image", lambda s: int(pd.Series(s).fillna(False).astype(bool).sum())),
            "primary_negative_count": ("negative_source_mode", lambda s: int((s == "primary_no_varroa_image").sum())),
            "fallback_negative_count": ("negative_source_mode", lambda s: int(s.astype(str).str.contains("fallback_varroa_image", na=False).sum())),
            "duplicate_fallback_count": ("flags", lambda s: int(s.astype(str).str.contains("duplicate_crop_fallback", na=False).sum())),
        }

        split_summary = (
            all_patch_metadata[patch_set]
            .groupby("split")
            .agg(**summary_agg)
            .reset_index()
        )

        for _, row in split_summary.iterrows():
            patch_metadata_manifest_records.append({
                "dataset_name": DATASET_NAME,
                "patch_set": patch_set,
                "patch_size": patch_size,
                "ratio_variant": ratio_variant,
                "split": row["split"],
                "total_rows": int(row["total_rows"]),
                "positive_count": int(row["positive_count"]),
                "negative_count": int(row["negative_count"]),
                "center_negative_count": int(row["center_negative_count"]),
                "mixed_negative_count": int(row["mixed_negative_count"]),
                "outer_negative_count": int(row["outer_negative_count"]),
                "shifted_crop_count": int(row["shifted_crop_count"]),
                "primary_negative_count": int(row["primary_negative_count"]),
                "fallback_negative_count": int(row["fallback_negative_count"]),
                "duplicate_fallback_count": int(row["duplicate_fallback_count"]),
                "metadata_path": str(output_path),
            })

patch_set_summary_df = pd.DataFrame(patch_metadata_manifest_records)

patch_metadata_manifest_df = (
    patch_set_summary_df
    .groupby(["dataset_name", "patch_set", "patch_size", "ratio_variant", "metadata_path"], dropna=False)
    .agg(
        total_rows=("total_rows", "sum"),
        positive_count=("positive_count", "sum"),
        negative_count=("negative_count", "sum"),
        center_negative_count=("center_negative_count", "sum"),
        mixed_negative_count=("mixed_negative_count", "sum"),
        outer_negative_count=("outer_negative_count", "sum"),
        shifted_crop_count=("shifted_crop_count", "sum"),
        primary_negative_count=("primary_negative_count", "sum"),
        fallback_negative_count=("fallback_negative_count", "sum"),
        duplicate_fallback_count=("duplicate_fallback_count", "sum"),
    )
    .reset_index()
)

patch_set_summary_df, _ = save_dataframe_csv(
    patch_set_summary_df,
    TABLES_DIR / "patch_set_summary.csv",
    overwrite=OVERWRITE_TABLES,
    note="Patch set split-level summary.",
)

patch_metadata_manifest_df, _ = save_dataframe_csv(
    patch_metadata_manifest_df,
    METADATA_DIR / "patch_metadata_manifest.csv",
    overwrite=OVERWRITE_PATCH_METADATA,
    file_type="patch_metadata_manifest",
    note="Dataset-level patch metadata manifest.",
)

log_dataframe("Patch Set Summary", patch_set_summary_df, max_rows=30)
log_dataframe("Patch Metadata Manifest", patch_metadata_manifest_df, max_rows=12)


## 11. Patch Metadata şeması

Üretilen metadata tablolarının kolon yapısı kontrol edilir.


In [ ]:
log_section("08 PATCH METADATA SCHEMA")

schema_records = []

column_descriptions = {
    "patch_id": "Deterministic unique patch id.",
    "dataset_name": "Dataset identifier.",
    "split": "train / valid / test split.",
    "source_image_path": "Raw source image path.",
    "source_label_path": "Raw source label path.",
    "patch_set": "Patch set name such as centered80_balanced.",
    "patch_size": "Square patch size in pixels.",
    "ratio_variant": "balanced or neg4x.",
    "patch_label": "1 positive, 0 negative.",
    "patch_type": "positive, center_negative, mixed_negative, outer_negative.",
    "x1": "Crop left coordinate.",
    "y1": "Crop top coordinate.",
    "x2": "Crop right coordinate.",
    "y2": "Crop bottom coordinate.",
    "crop_width": "Crop width in pixels.",
    "crop_height": "Crop height in pixels.",
    "image_width": "Source image width.",
    "image_height": "Source image height.",
    "patch_center_x_px": "Final crop center x after boundary shift.",
    "patch_center_y_px": "Final crop center y after boundary shift.",
    "requested_center_x_px": "Requested crop center x before boundary shift.",
    "requested_center_y_px": "Requested crop center y before boundary shift.",
    "was_shifted_to_fit_image": "Whether crop was shifted to fit image boundaries.",
    "shift_x_px": "Crop shift on x axis.",
    "shift_y_px": "Crop shift on y axis.",
    "source_bbox_id": "Source bbox id for positive rows.",
    "source_bbox_row_index": "YOLO row index for positive rows.",
    "source_bbox_class_id": "Source bbox class id for positive rows.",
    "source_bbox_class_name": "Source bbox class name for positive rows.",
    "source_bbox_x_center_norm": "YOLO x center normalized.",
    "source_bbox_y_center_norm": "YOLO y center normalized.",
    "source_bbox_width_norm": "YOLO bbox width normalized.",
    "source_bbox_height_norm": "YOLO bbox height normalized.",
    "source_bbox_x_center_px": "BBox x center in pixels.",
    "source_bbox_y_center_px": "BBox y center in pixels.",
    "source_bbox_width_px": "BBox width in pixels.",
    "source_bbox_height_px": "BBox height in pixels.",
    "source_bbox_area_px": "BBox area in pixels.",
    "is_valid_yolo_row": "Whether source YOLO row was valid.",
    "is_extreme_small_candidate": "Dataset-specific small bbox outlier flag.",
    "is_extreme_large_candidate": "Dataset-specific large bbox outlier flag.",
    "is_bbox_exclusion_candidate": "Final bbox-level exclusion flag.",
    "negative_type": "Negative type for negative rows.",
    "requested_negative_zone": "Requested negative zone before crop shift.",
    "actual_negative_zone": "Actual zone after crop shift.",
    "actual_normalized_center_distance": "Actual normalized distance from image center.",
    "negative_source_mode": "primary_no_varroa_image or fallback varroa-image non-overlap mode.",
    "max_bbox_inside_patch_ratio": "Maximum fraction of any Varroa bbox area covered by the negative patch.",
    "overlaps_varroa_bbox": "Whether the negative patch crosses the accepted overlap threshold.",
    "sampling_seed": "Sampling seed.",
    "sampling_attempt_index": "Attempt index used for negative sampling.",
    "notes": "Free text notes.",
    "flags": "Semicolon-separated technical flags.",
}

for column in PATCH_METADATA_COLUMNS:
    example_dtype = ""
    for patch_df in all_patch_metadata.values():
        if column in patch_df.columns:
            example_dtype = str(patch_df[column].dtype)
            break

    schema_records.append({
        "column_name": column,
        "dtype": example_dtype,
        "nullable": True,
        "description": column_descriptions.get(column, ""),
    })

patch_metadata_schema_df = pd.DataFrame(schema_records)

patch_metadata_schema_df, _ = save_dataframe_csv(
    patch_metadata_schema_df,
    TABLES_DIR / "patch_metadata_schema.csv",
    overwrite=OVERWRITE_TABLES,
    note="Patch metadata schema definition.",
)

log_dataframe("Patch Metadata Schema", patch_metadata_schema_df, max_rows=60)


## 12. Görsel Kontrol Fonksiyonları

Sanity-check görsellerinde patch ve bbox alanlarını çizmek için yardımcı fonksiyonlar hazırlanır.


In [ ]:
log_section("09 SANITY-CHECK HELPER FONKSİYONLARI")

def draw_source_with_boxes(row, max_thumbnail_size=220):
    image_path = Path(row["source_image_path"])

    with Image.open(image_path).convert("RGB") as image:
        original_width, original_height = image.size
        scale = min(max_thumbnail_size / original_width, max_thumbnail_size / original_height)
        thumbnail_size = (int(original_width * scale), int(original_height * scale))
        thumbnail = image.resize(thumbnail_size)

    draw = ImageDraw.Draw(thumbnail)

    # Kırmızı: patch crop box
    patch_box = [
        int(row["x1"] * scale),
        int(row["y1"] * scale),
        int(row["x2"] * scale),
        int(row["y2"] * scale),
    ]
    draw.rectangle(patch_box, outline="red", width=3)

    # Yeşil: source Varroa bbox, only for positive rows
    if int(row["patch_label"]) == 1 and pd.notna(row["source_bbox_x_center_px"]):
        bbox_x_center = float(row["source_bbox_x_center_px"])
        bbox_y_center = float(row["source_bbox_y_center_px"])
        bbox_width = float(row["source_bbox_width_px"])
        bbox_height = float(row["source_bbox_height_px"])

        bbox_box = [
            int((bbox_x_center - bbox_width / 2) * scale),
            int((bbox_y_center - bbox_height / 2) * scale),
            int((bbox_x_center + bbox_width / 2) * scale),
            int((bbox_y_center + bbox_height / 2) * scale),
        ]

        draw.rectangle(bbox_box, outline="lime", width=3)

    return thumbnail


def crop_patch(row):
    image_path = Path(row["source_image_path"])

    with Image.open(image_path).convert("RGB") as image:
        patch = image.crop((int(row["x1"]), int(row["y1"]), int(row["x2"]), int(row["y2"])))

    return patch


def make_sanity_grid(rows_df, output_path, title, max_examples=12):
    if rows_df.empty:
        log_output(f"Sanity grid için örnek yok: {title}", level="WARNING")
        return None

    selected_df = rows_df.head(max_examples).copy()

    n_examples = len(selected_df)
    n_cols = 4
    n_rows = math.ceil(n_examples / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(n_cols * 5.2, n_rows * 3.8),
        squeeze=False,
    )

    for ax in axes.ravel():
        ax.axis("off")

    for ax, (_, row) in zip(axes.ravel(), selected_df.iterrows()):
        source_thumb = draw_source_with_boxes(row)
        patch = crop_patch(row).resize((source_thumb.width, source_thumb.height))

        combined = Image.new("RGB", (source_thumb.width * 2 + 8, source_thumb.height), color="white")
        combined.paste(source_thumb, (0, 0))
        combined.paste(patch, (source_thumb.width + 8, 0))

        ax.imshow(combined)
        ax.axis("off")

        caption = (
            f"{Path(row['source_image_path']).name}\n"
            f"{row['split']} | {int(row['patch_size'])}x{int(row['patch_size'])} | {row['patch_type']}\n"
            f"shifted={row['was_shifted_to_fit_image']}"
        )

        if row["patch_label"] == 0:
            dist_value = row.get("actual_normalized_center_distance", np.nan)
            if pd.notna(dist_value):
                caption += (
                    f"\nreq={row['requested_negative_zone']} | actual={row['actual_negative_zone']} | dist={float(dist_value):.2f}"
                )
            else:
                caption += (
                    f"\nreq={row['requested_negative_zone']} | actual={row['actual_negative_zone']}"
                )

        ax.set_title(caption, fontsize=8)

    fig.suptitle(title + "\nRed: patch crop box | Green: Varroa bbox", fontsize=14)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not OVERWRITE_FIGURES:
        plt.close(fig)
        log_output(f"[SKIP] Existing figure kept: {relative_path(output_path)}")
        return output_path

    if output_path.exists() and OVERWRITE_FIGURES:
        log_output(f"[OVERWRITE] Replacing figure: {relative_path(output_path)}")

    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    log_output(f"[SAVE] Figure saved: {relative_path(output_path)}")
    return output_path


## 13. Sanity-Check Görselleri

Her split ve patch boyutu için örnek pozitif ve negatif patch görselleri üretilir.


In [ ]:
log_section("10 SANITY-CHECK GÖRSELLERİ")

sanity_records = []

for patch_size in PATCH_SIZES:
    reference_patch_set = f"centered{patch_size}_balanced"

    if reference_patch_set not in all_patch_metadata:
        log_output(f"Reference patch set bulunamadı: {reference_patch_set}", level="WARNING")
        continue

    reference_df = all_patch_metadata[reference_patch_set].copy()

    for split in SPLITS:
        split_df = reference_df[reference_df["split"] == split].copy()

        positive_source = split_df[split_df["patch_type"] == "positive"].copy()
        positive_examples = (
            positive_source.sample(
                n=min(SANITY_EXAMPLES_PER_GRID, len(positive_source)),
                random_state=RANDOM_STATE,
            )
            if len(positive_source) > 0
            else pd.DataFrame()
        )

        positive_output_path = FIGURES_DIR / f"sanity_{DATASET_SHORT_NAME}_{split}_{patch_size}_positive.png"

        if not positive_examples.empty:
            make_sanity_grid(
                positive_examples,
                positive_output_path,
                f"{DATASET_NAME} | {split} | {patch_size}x{patch_size} | positive",
                max_examples=SANITY_EXAMPLES_PER_GRID,
            )

            for _, row in positive_examples.iterrows():
                sanity_records.append({
                    "dataset_name": DATASET_NAME,
                    "split": split,
                    "patch_size": patch_size,
                    "patch_type": "positive",
                    "source_image_path": row["source_image_path"],
                    "figure_path": str(positive_output_path),
                })

        for negative_type in NEGATIVE_TYPES:
            negative_examples_source = split_df[split_df["patch_type"] == negative_type].copy()

            if negative_examples_source.empty:
                log_output(
                    f"Sanity negative örneği yok: {split} | {patch_size} | {negative_type}",
                    level="WARNING",
                )
                continue

            negative_examples = negative_examples_source.sample(
                n=min(SANITY_EXAMPLES_PER_GRID, len(negative_examples_source)),
                random_state=RANDOM_STATE,
            )

            negative_output_path = FIGURES_DIR / f"sanity_{DATASET_SHORT_NAME}_{split}_{patch_size}_{negative_type}.png"

            make_sanity_grid(
                negative_examples,
                negative_output_path,
                f"{DATASET_NAME} | {split} | {patch_size}x{patch_size} | {negative_type}",
                max_examples=SANITY_EXAMPLES_PER_GRID,
            )

            for _, row in negative_examples.iterrows():
                sanity_records.append({
                    "dataset_name": DATASET_NAME,
                    "split": split,
                    "patch_size": patch_size,
                    "patch_type": negative_type,
                    "source_image_path": row["source_image_path"],
                    "figure_path": str(negative_output_path),
                })

sanity_check_examples_df = pd.DataFrame(sanity_records)

sanity_check_examples_df, _ = save_dataframe_csv(
    sanity_check_examples_df,
    TABLES_DIR / "sanity_check_examples.csv",
    overwrite=OVERWRITE_TABLES,
    note="Sanity-check figure example records.",
)

log_dataframe("Sanity Check Examples", sanity_check_examples_df, max_rows=20)


## 14. Doğrulama Kontrolleri

Patch oranları, koordinat sınırları ve negatif örnek overlap değerleri kontrol edilir.


In [ ]:
log_section("11 VALIDATION CHECKS")

validation_records = []

for patch_set, patch_df in all_patch_metadata.items():
    for split in SPLITS:
        split_df = patch_df[patch_df["split"] == split]

        positive_count = int((split_df["patch_label"] == 1).sum())
        negative_count = int((split_df["patch_label"] == 0).sum())

        ratio_variant = split_df["ratio_variant"].dropna().iloc[0] if not split_df.empty else ""

        if ratio_variant == "balanced":
            expected_negative_count = positive_count
        elif ratio_variant == "neg4x":
            expected_negative_count = positive_count * 4
        else:
            expected_negative_count = np.nan

        ratio_ok = (
            pd.notna(expected_negative_count)
            and negative_count == int(expected_negative_count)
        )

        coordinate_ok = True
        overlap_ok = True
        max_negative_overlap = 0.0

        if not split_df.empty:
            coordinate_ok = bool((
                (split_df["x1"] >= 0)
                & (split_df["y1"] >= 0)
                & (split_df["x2"] <= split_df["image_width"])
                & (split_df["y2"] <= split_df["image_height"])
                & (split_df["x2"] > split_df["x1"])
                & (split_df["y2"] > split_df["y1"])
            ).all())

            negative_df = split_df[split_df["patch_label"] == 0].copy()
            if "max_bbox_inside_patch_ratio" in negative_df.columns and not negative_df.empty:
                max_negative_overlap = float(pd.to_numeric(
                    negative_df["max_bbox_inside_patch_ratio"], errors="coerce"
                ).fillna(0).max())
                overlap_ok = bool(max_negative_overlap < MAX_BBOX_INSIDE_PATCH_RATIO)

        validation_records.append({
            "dataset_name": DATASET_NAME,
            "patch_set": patch_set,
            "split": split,
            "positive_count": positive_count,
            "negative_count": negative_count,
            "expected_negative_count": expected_negative_count,
            "ratio_ok": bool(ratio_ok),
            "coordinate_ok": bool(coordinate_ok),
            "overlap_ok": bool(overlap_ok),
            "max_negative_bbox_inside_patch_ratio": max_negative_overlap,
            "status": "OK" if ratio_ok and coordinate_ok and overlap_ok else "CHECK",
        })

validation_checks_df = pd.DataFrame(validation_records)

validation_checks_df, _ = save_dataframe_csv(
    validation_checks_df,
    TABLES_DIR / "validation_checks.csv",
    overwrite=OVERWRITE_TABLES,
    note="Patch metadata validation checks.",
)

log_dataframe("Validation Checks", validation_checks_df, max_rows=30)


## 15. Final Durum

Notebook sonunda üretilen metadata dosyaları ve doğrulama sonuçları özetlenir.


In [ ]:
log_section("12 FINAL DURUM")

problem_count = int((validation_checks_df["status"] != "OK").sum())
metadata_file_count = len(all_patch_metadata)
sanity_figure_count = int(sanity_check_examples_df["figure_path"].nunique()) if not sanity_check_examples_df.empty else 0

if metadata_file_count != len(PATCH_SIZES) * len(RATIO_VARIANTS):
    final_status = "FAILED"
elif problem_count > 0:
    final_status = "NEEDS_MANUAL_REVIEW"
else:
    final_status = "READY_FOR_NEXT_STAGE"

notebook_status_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "stage_name": STAGE_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "final_status": final_status,
    "metadata_file_count": metadata_file_count,
    "validation_problem_count": problem_count,
    "sanity_figure_count": sanity_figure_count,
    "raw_data_modified": False,
    "run_timestamp": RUN_TIMESTAMP,
}])

notebook_status_df, _ = save_dataframe_csv(
    notebook_status_df,
    TABLES_DIR / "notebook_status.csv",
    overwrite=OVERWRITE_TABLES,
    note="Notebook final status.",
)

log_dataframe("Notebook Status", notebook_status_df)
log_output(f"Final status: {final_status}")
log_output("Patch preparation notebook tamamlandı.")
